# 03 - Contr?le qualit? des donn?es

Ce notebook ex?cute un contr?le qualit? complet de la base brute, sans corriger silencieusement quoi que ce soit.

Il utilise les fonctions r?utilisables de `src/quality` pour d?tecter :

- les doublons et identifiants dupliqu?s ;
- les valeurs manquantes (nombre et pourcentage) ;
- les incoh?rences de type ;
- les cat?gories inattendues ;
- les valeurs impossibles ;
- les valeurs aberrantes (m?thode IQR) ;
- les distributions suspectes ;
- la coh?rence entre variables li?es.

Chaque probl?me est class? en **CRITIQUE**, **IMPORTANT** ou **MINEUR**. Une synth?se est enregistr?e dans `outputs/reports/`.

**Aucune correction n'est appliqu?e dans ce notebook.** Les d?cisions de correction sont uniquement document?es ; elles seront mises en ?uvre et justifi?es dans `04_nettoyage.ipynb`.


In [1]:
from src.data import load_raw_dataset
from src.quality import run_full_quality_check, render_report_markdown, save_quality_report
from src.utils.paths import REPORTS_DIR

result = load_raw_dataset()
df = result.dataframe
print('Fichier analys? :', result.path)
print('Dimensions :', df.shape)


Fichier analys? : C:\Users\admin\Desktop\PROJET STATISTIQUE PUBLIC B\data\raw\donnees_brutes_education_prioritaire.csv
Dimensions : (5250, 22)


## Nombre de lignes et de colonnes


In [2]:
report = run_full_quality_check(df)
print('Nombre de lignes :', report.n_rows)
print('Nombre de colonnes :', report.n_columns)
print('Synth?se par s?v?rit? :', report.summary_counts())


Nombre de lignes : 5250
Nombre de colonnes : 22
Synth?se par s?v?rit? : {'CRITIQUE': 0, 'IMPORTANT': 1, 'MINEUR': 10}


## Doublons et identifiants dupliqu?s


In [3]:
from src.quality import check_duplicate_rows, check_duplicate_identifiers

dup_rows = check_duplicate_rows(df)
dup_ids = check_duplicate_identifiers(df)
print(dup_rows.to_dict())
print(dup_ids.to_dict())


{'code': 'DUPLICATE_ROWS', 'severity': 'MINEUR', 'variable': None, 'description': "Lignes strictement identiques sur l'ensemble des colonnes.", 'n_affected': 0, 'details': {'pourcentage': 0.0}}
{'code': 'DUPLICATE_IDENTIFIERS', 'severity': 'MINEUR', 'variable': 'annee, ecole_id, niveau', 'description': "Combinaisons d'identifiant logique (année, école, niveau) apparaissant plusieurs fois.", 'n_affected': 0, 'details': {'colonnes_utilisees': ['annee', 'ecole_id', 'niveau']}}


## Valeurs manquantes par variable (nombre et pourcentage)


In [4]:
import pandas as pd

na_count = df.isna().sum()
na_pct = (100 * na_count / len(df)).round(3)
na_table = pd.DataFrame({'n_manquants': na_count, 'pourcentage_manquant': na_pct})
na_table = na_table[na_table['n_manquants'] > 0].sort_values('pourcentage_manquant', ascending=False)
if na_table.empty:
    print('Aucune valeur manquante d?tect?e sur les colonnes de la base.')
else:
    display(na_table)


Aucune valeur manquante d?tect?e sur les colonnes de la base.


## Anomalies : types, cat?gories inattendues, valeurs impossibles


In [5]:
from src.quality import check_type_consistency, check_unexpected_categories, check_impossible_values

for issue in check_type_consistency(df):
    print(issue.severity, '-', issue.code, '-', issue.variable, '-', issue.description)
for issue in check_unexpected_categories(df):
    print(issue.severity, '-', issue.code, '-', issue.variable, '-', issue.description)
for issue in check_impossible_values(df):
    print(issue.severity, '-', issue.code, '-', issue.variable, '-', issue.description)


## Valeurs aberrantes (m?thode IQR)


In [6]:
from src.quality import check_outliers_iqr

outlier_issues = check_outliers_iqr(df)
if not outlier_issues:
    print("Aucune valeur aberrante d?tect?e par la m?thode IQR sur les colonnes num?riques.")
else:
    for issue in outlier_issues:
        print(f"{issue.severity} - {issue.variable} - {issue.n_affected} observations - bornes {issue.details['borne_basse']} / {issue.details['borne_haute']}")


MINEUR - effectif_eleves - 10 observations - bornes 4.0 / 36.0
MINEUR - taille_moyenne_classe - 10 observations - bornes 3.055 / 29.575
MINEUR - score_francais - 7 observations - bornes 49.812 / 108.433
MINEUR - score_mathematiques - 16 observations - bornes 46.095 / 108.295
MINEUR - score_global - 8 observations - bornes 48.175 / 108.135
MINEUR - taux_maitrise_francais - 15 observations - bornes 50.62 / 96.3
MINEUR - taux_maitrise_mathematiques - 15 observations - bornes 46.916 / 94.026
MINEUR - variable_cible - 8 observations - bornes 48.175 / 108.135


## Distributions suspectes et coh?rence entre variables


In [7]:
from src.quality import check_suspicious_distributions, check_cross_variable_consistency

for issue in check_suspicious_distributions(df):
    print(issue.severity, '-', issue.code, '-', issue.variable, '-', issue.description)
print('---')
for issue in check_cross_variable_consistency(df):
    print(issue.severity, '-', issue.code, '-', issue.variable, '-', issue.description)


IMPORTANT - ZERO_VARIANCE - nombre_classes - La variable 'nombre_classes' ne prend qu'une seule valeur (variance nulle).
---


## Rapport complet et enregistrement

Le rapport complet, avec tous les probl?mes class?s par s?v?rit?, est affich? ci-dessous puis enregistr? dans `outputs/reports/`.


In [8]:
markdown_report = render_report_markdown(report, result.path)
print(markdown_report)


# Rapport de contrôle qualité

Généré le : 2026-09-03T00:09:26.685679+00:00
Fichier analysé : C:\Users\admin\Desktop\PROJET STATISTIQUE PUBLIC B\data\raw\donnees_brutes_education_prioritaire.csv

## Vue d'ensemble

- Nombre de lignes : 5250
- Nombre de colonnes : 22
- Problèmes CRITIQUES : 0
- Problèmes IMPORTANTS : 1
- Problèmes MINEURS : 10

## Détail des problèmes détectés

### IMPORTANT

- **ZERO_VARIANCE** (variable : nombre_classes) — La variable 'nombre_classes' ne prend qu'une seule valeur (variance nulle). — observations concernées : 5250
    - valeur_unique : 2.0

### MINEUR

- **DUPLICATE_ROWS** — Lignes strictement identiques sur l'ensemble des colonnes. — observations concernées : 0
    - pourcentage : 0.0
- **DUPLICATE_IDENTIFIERS** (variable : annee, ecole_id, niveau) — Combinaisons d'identifiant logique (année, école, niveau) apparaissant plusieurs fois. — observations concernées : 0
    - colonnes_utilisees : ['annee', 'ecole_id', 'niveau']
- **OUTLIER_IQR** (variable 

In [9]:
saved_paths = save_quality_report(report, REPORTS_DIR, dataset_path=result.path)
print('Rapport Markdown enregistr? :', saved_paths['markdown'])
print('Rapport JSON enregistr? :', saved_paths['json'])


Rapport Markdown enregistr? :

 C:\Users\admin\Desktop\PROJET STATISTIQUE PUBLIC B\outputs\reports\rapport_qualite_donnees_brutes.md
Rapport JSON enregistr? : C:\Users\admin\Desktop\PROJET STATISTIQUE PUBLIC B\outputs\reports\rapport_qualite_donnees_brutes.json


## D?cisions de correction (? justifier, pas encore appliqu?es)

Aucune correction n'est appliqu?e dans ce notebook. Sur la base des probl?mes d?tect?s ci-dessus, les d?cisions envisag?es pour `04_nettoyage.ipynb` sont, ? titre d'exemple :

- **Doublons de lignes** : ? supprimer si confirm?s, en conservant une trace du nombre de lignes supprim?es et de la r?gle utilis?e (garder la premi?re occurrence).
- **Identifiants dupliqu?s** : ? investiguer avant toute action ; une duplication d'identifiant logique (ann?e, ?cole, niveau) peut signaler soit une vraie erreur de collecte, soit une granularit? de donn?es mal comprise (ex. plusieurs classes par niveau).
- **Valeurs manquantes** : la strat?gie (suppression, imputation, ou conservation avec indicateur de manquant) doit d?pendre du pourcentage de valeurs manquantes et du r?le de la variable dans l'analyse ; aucune imputation ne doit ?tre faite sans justification explicite.
- **Valeurs impossibles / cat?gories inattendues** : ? corriger uniquement apr?s v?rification de la source ; si aucune source de r?f?rence n'est disponible, les lignes concern?es seront isol?es et document?es plut?t que supprim?es automatiquement.
- **Valeurs aberrantes (IQR)** : ne seront pas supprim?es par d?faut ; une valeur aberrante statistique n'est pas n?cessairement une erreur de donn?es (ex. un tr?s petit ?tablissement).
- **Distributions suspectes (variance nulle, concentration excessive)** : ? examiner variable par variable pour d?terminer si la variable reste informative pour la suite de l'analyse ou si elle doit ?tre ?cart?e de la mod?lisation.
- **Incoh?rences entre variables** : les lignes concern?es devront ?tre corrig?es de fa?on transparente (avec tra?age de la r?gle de recalcul appliqu?e), et non simplement supprim?es.

Chaque correction effectu?e dans le notebook de nettoyage devra indiquer explicitement : la r?gle appliqu?e, le nombre de lignes concern?es, et la justification m?tier ou statistique.
